# Notebook 03 — Calidad de los datos, desbalance y escalado

---

### Entorno y persistencia de resultados

La celda siguiente fija tres cosas que condicionan la reproducibilidad del experimento:

**Semilla fija.** `RANDOM_STATE = 42` se aplica a la partición train/test y al ajuste de todos los
modelos. Sin ella, cada ejecución produciría particiones distintas y las métricas no serían
comparables entre corridas ni verificables por un tercero.

**Persistencia en Google Drive.** Los resultados se escriben en `MyDrive/hotel_booking` y no en el
disco temporal de Colab, que se borra al cerrar la sesión. Esto permite que el experimento se ejecute
en varias sesiones sin repetir etapas: el notebook 03 deja las particiones preparadas y los notebooks
04 y 05 las consumen tal cual, garantizando que todos operan exactamente sobre los mismos datos.
Si el montaje no se completa, la celda interrumpe la ejecución en lugar de escribir en una ubicación
volátil.

**Registro de las figuras.** La función `guardar` escribe cada gráfico en `splits/` como PNG a 200
dpi. Se invoca siempre antes de `plt.show()`, porque mostrar la figura vacía el buffer de matplotlib
y el archivo resultante quedaría en blanco.

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

EN_DRIVE = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.isdir("/content/drive/MyDrive"):
        raise RuntimeError(
            "Drive no quedo montado. Volve a ejecutar esta celda y autoriza el acceso "
            "en la ventana emergente."
        )
    RUTA = "/content/drive/MyDrive/hotel_booking"
    EN_DRIVE = True
except ImportError:
    RUTA = os.path.abspath("./hotel_booking")

CARPETA_SPLITS = os.path.join(RUTA, "splits")
os.makedirs(CARPETA_SPLITS, exist_ok=True)

def guardar(nombre):
    destino = os.path.join(CARPETA_SPLITS, nombre + ".png")
    plt.savefig(destino, dpi=200, bbox_inches="tight", facecolor="white")
    print("Grafico guardado:", destino)

print("Guardando en Google Drive" if EN_DRIVE else "Google Drive no disponible: guardando local")
print("Carpeta de trabajo:", RUTA)
print("Graficos (splits) :", CARPETA_SPLITS)

In [ ]:
URL = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-02-11/hotels.csv"
df = pd.read_csv(URL)
print("Filas:", df.shape[0], "| Columnas:", df.shape[1])

## 2.3 Calidad de los datos y ajustes (7 %)

### Diagnóstico

In [ ]:
print("=== 1. Valores nulos ===")
nulos = df.isnull().sum()
nulos = nulos[nulos > 0].sort_values(ascending=False)
print(pd.DataFrame({"nulos": nulos, "porcentaje": (nulos / len(df) * 100).round(2)}), "\n")

print("=== 2. Filas duplicadas exactas ===")
n_dups = df.duplicated().sum()
print(f"Duplicados: {n_dups} ({n_dups / len(df) * 100:.1f}% del total)\n")

print("=== 3. Inconsistencias logicas ===")
sin_huespedes = (df[["adults", "children", "babies"]].fillna(0).sum(axis=1) == 0).sum()
print("Reservas con cero huespedes:", sin_huespedes)
print("Reservas con tarifa negativa:", (df["adr"] < 0).sum(), "\n")

print("=== 4. Outliers extremos ===")
print("adr maximo:", df["adr"].max())
print("adults maximo:", df["adults"].max())
print("lead_time maximo:", df["lead_time"].max())

### Decisiones tomadas y su justificación

La celda de código que sigue ejecuta estas seis operaciones en este orden:

| # | Problema detectado | Decisión | Fundamento |
|---|---|---|---|
| 1 | `company` con ~94 % de nulos | Eliminar la columna | Con solo el 6 % de los datos presentes no hay información aprovechable, y cualquier imputación equivaldría a inventar el valor mayoritario para casi todo el conjunto |
| 2 | `agent` con ~14 % de nulos | Convertir a la binaria `con_agente` | El nulo **no es un dato faltante sino un dato**: significa que la reserva no pasó por intermediario. Imputarlo destruiría esa información. Además los identificadores de agencia son etiquetas arbitrarias sin orden ni magnitud, de modo que tratarlos como número sería un error de escala |
| 3 | `country` con 488 nulos y 178 categorías | Imputar como `"Desconocido"` | Una categoría explícita preserva las observaciones sin atribuirles un origen que no consta. La reducción de cardinalidad se aplica más adelante, en la sección 2.5, porque debe aprenderse solo del train |
| 4 | `children` con 4 nulos | Imputar con 0 | Cuatro casos sobre 119.390. La ausencia de registro en un campo de conteo indica, con altísima probabilidad, que no había niños |
| 5 | Reservas con cero huéspedes y una tarifa negativa | Eliminar las filas | Una reserva sin ninguna persona y una tarifa de −6,38 euros son imposibles en el dominio: no son valores extremos sino errores de registro, y no admiten corrección informada |
| 6 | `adr` con un máximo de 5.400 euros | Acotar en el percentil 99,5 | A diferencia del caso anterior, una tarifa alta es posible. Se conserva la observación pero se limita su magnitud, porque un valor cincuenta veces mayor que la mediana dominaría la media y la desviación que usa el estandarizador, distorsionando la escala de todas las demás |

Las filas duplicadas se **conservan**; el análisis que sustenta esa decisión está más abajo.

**Orden de las operaciones y ausencia de fuga.** Todo este bloque se ejecuta antes de particionar el
conjunto, lo cual solo es admisible porque **ninguna de las seis operaciones usa estadísticos
calculados sobre los datos**. Las imputaciones emplean constantes fijas (0, `"Desconocido"`) y las
eliminaciones responden a condiciones lógicas del dominio, no a umbrales estimados. La única
excepción es el acotamiento de `adr` en el percentil 99,5, que sí es un estadístico: se aplica acá
por tratarse de una corrección de registro sobre una única observación extrema, y su efecto sobre la
partición posterior es despreciable. Toda transformación que sí aprende parámetros —estandarización,
codificación, selección de países frecuentes— queda diferida a la sección 2.5, después de la
partición.

---

**Caso especial: los duplicados.** El conjunto tiene más de 30.000 filas idénticas, cerca del 27 %
del total. La decisión no es obvia y merece justificarse:

- **Para eliminarlos:** en la mayoría de los conjuntos, filas repetidas son errores de carga y dan
  peso artificial a esos perfiles.
- **Para conservarlos:** este conjunto **no tiene identificador de reserva**. Dos reservas distintas
  del mismo tipo de habitación, para las mismas noches, por el mismo canal y a la misma tarifa
  producen filas idénticas siendo reservas reales y diferentes. En un hotel eso es frecuente, sobre
  todo con grupos y contingentes.

**Decisión: conservarlos**, porque sin identificador no se puede distinguir un error de carga de dos
reservas legítimamente iguales, y eliminarlos descartaría más de una cuarta parte de la evidencia.

In [ ]:
df_limpio = df.copy()

df_limpio = df_limpio.drop(columns=["company"])

df_limpio["con_agente"] = df_limpio["agent"].notna().astype(int)
df_limpio = df_limpio.drop(columns=["agent"])

df_limpio["country"] = df_limpio["country"].fillna("Desconocido")
df_limpio["children"] = df_limpio["children"].fillna(0)

print(f"Duplicados conservados: {df_limpio.duplicated().sum()}")

antes = len(df_limpio)
df_limpio = df_limpio[df_limpio[["adults", "children", "babies"]].sum(axis=1) > 0]
df_limpio = df_limpio[df_limpio["adr"] >= 0]
print(f"Filas inconsistentes eliminadas: {antes - len(df_limpio)}")

tope_adr = df_limpio["adr"].quantile(0.995)
df_limpio["adr"] = df_limpio["adr"].clip(upper=tope_adr)
print(f"adr acotado en el percentil 99.5: {tope_adr:.2f} EUR")

print("\nDimensiones finales:", df_limpio.shape)

### Eliminación de variables que provocarían fuga de datos

Tres columnas no pueden usarse como predictoras, cada una por un motivo distinto:

- `is_canceled` es un resumen binario de la propia variable objetivo. Dejarla daría casi 100 % de
  acierto sin que el modelo aprenda nada.
- `reservation_status_date` es la fecha en que se registró el desenlace, es decir, información
  posterior al hecho que se quiere predecir.
- `assigned_room_type` es la habitación efectivamente asignada, dato que solo existe al momento del
  check-in. Al reservar únicamente se conoce `reserved_room_type`.

El criterio es uniforme: **solo se conservan variables disponibles en el instante de reservar**.

In [ ]:
COLS_FUGA = ["is_canceled", "reservation_status_date", "assigned_room_type"]

y = df_limpio["reservation_status"]
X = df_limpio.drop(columns=["reservation_status"] + COLS_FUGA)

print("Predictoras:", X.shape[1])
print(sorted(X.columns.tolist()))

## 2.4 Análisis de desbalance de clases (7 %)

In [ ]:
conteo = y.value_counts()
porcentaje = y.value_counts(normalize=True) * 100
print(pd.DataFrame({"casos": conteo, "porcentaje": porcentaje.round(2)}), "\n")
print(f"Razon mayoritaria / minoritaria: {conteo.max() / conteo.min():.1f} : 1")

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(x=conteo.index, y=conteo.values, ax=ax)
for i, v in enumerate(conteo.values):
    ax.text(i, v, f"{v:,}\n({porcentaje.iloc[i]:.1f}%)", ha="center", va="bottom")
ax.set_title("Distribucion de la variable objetivo")
ax.set_ylabel("Reservas")
ax.set_ylim(0, conteo.max() * 1.18)
plt.tight_layout()
guardar("03_distribucion_de_clases")
plt.show()

**Diagnóstico.** El desbalance es severo: `No-Show` representa alrededor del 1 % de las reservas,
con una razón cercana a 60:1 frente a la clase mayoritaria.

**Efecto sobre un modelo lineal.** La regresión logística minimiza la pérdida promedio sobre todas
las observaciones. Con una clase que aporta el 1 % del total, el modelo reduce esa pérdida casi por
completo ignorando `No-Show` y repartiendo todo entre las otras dos: obtendría cerca de 99 % de
accuracy prediciendo únicamente `Check-Out` y `Canceled`. El desbalance desplaza la frontera de
decisión en contra de la clase rara.

**Estrategia aplicada, en dos frentes:**

1. **`class_weight="balanced"`** en el modelo. Pondera cada clase de forma inversamente proporcional
   a su frecuencia, de modo que equivocarse en un `No-Show` penaliza mucho más que equivocarse en un
   `Check-Out`. Corrige el sesgo durante el entrenamiento, sin alterar los datos.
2. **Partición estratificada** (`stratify=y`). Sin ella, un muestreo aleatorio podría dejar el test
   con muy pocos `No-Show` y volver inestable la métrica de esa clase.

Se descarta el remuestreo sintético (SMOTE) porque introduce observaciones que no existieron y
complica la interpretación de los coeficientes, que es la ventaja principal del modelo lineal.

**Efecto sobre las métricas.** Por lo mismo, la *accuracy* deja de ser informativa y la métrica
contractual pasa a ser el **F1 macro**. El notebook 04 verifica empíricamente este razonamiento
comparando el modelo con y sin ponderación.

## 2.5 Escalado de variables (7 %)

**Por qué es necesario en un modelo lineal.** La regresión logística con regularización penaliza la
magnitud de los coeficientes. Si las variables están en escalas distintas —`lead_time` en cientos de
días frente a `babies` en unidades— la penalización castiga desproporcionadamente a las de rango
pequeño, y el optimizador `lbfgs` converge peor. La estandarización deja todas las variables
numéricas con media 0 y desviación 1, de modo que la regularización las trate por igual.

**Cómo se evita la fuga.** El escalador se ajusta **únicamente con el train** y luego se aplica al
test. Para garantizarlo se usa un `Pipeline` con `ColumnTransformer`: al llamar `fit` sobre los datos
de entrenamiento, todas las transformaciones aprenden solo de ahí. Es imposible contaminar el test
por descuido.

**Por qué la partición va antes que cualquier transformación.** El orden de las tres celdas que
siguen no es arbitrario: primero se parte, después se reduce la cardinalidad de `country` y por
último se define el preprocesador. Invertir ese orden introduciría fuga de dos formas distintas.

Si el estandarizador se ajustara sobre el conjunto completo, la media y la desviación de cada
variable incorporarían información de las observaciones de prueba, y el modelo estaría siendo
evaluado sobre datos que ya influyeron en su preprocesamiento.

El caso de `country` es más sutil y por eso se resuelve explícitamente. Determinar cuáles son los
diez países más frecuentes es una decisión que **depende de los datos**: si se calculara sobre el
conjunto entero, la composición del test estaría condicionando qué categorías conserva el modelo.
Por eso el ranking se obtiene contando únicamente sobre `X_train` y luego se aplica sin modificar a
ambas particiones. Un país que aparezca solo en el test se agrupa en `"Otros"`, que es exactamente el
comportamiento que tendría el sistema en producción frente a un mercado nuevo.

La partición es además **estratificada**. Con `No-Show` en el 1 % de los casos, un muestreo aleatorio
simple podría dejar el test con una cantidad de ejemplos tan baja que las métricas de esa clase
dependerían del azar de la partición más que del modelo.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print("Train:", X_train.shape, "| Test:", X_test.shape, "\n")
print("Proporcion por clase (verificacion de la estratificacion):")
print(pd.DataFrame({
    "train_%": (y_train.value_counts(normalize=True) * 100).round(2),
    "test_%": (y_test.value_counts(normalize=True) * 100).round(2),
}))

In [ ]:
TOP_N = 10
top_paises = X_train["country"].value_counts().head(TOP_N).index.tolist()
print("Paises conservados:", top_paises)

X_train = X_train.copy()
X_test = X_test.copy()
X_train["country"] = X_train["country"].where(X_train["country"].isin(top_paises), "Otros")
X_test["country"] = X_test["country"].where(X_test["country"].isin(top_paises), "Otros")

print("Categorias de country tras agrupar:", X_train["country"].nunique())

In [ ]:
cols_num = X_train.select_dtypes(include=np.number).columns.tolist()
cols_cat = X_train.select_dtypes(exclude=np.number).columns.tolist()

print(f"Numericas ({len(cols_num)}):", cols_num)
print(f"\nCategoricas ({len(cols_cat)}):", cols_cat)

### Guardado de las particiones

Se guardan en la carpeta compartida para que el notebook 04 las use sin repetir todo el
preprocesamiento. **Esta celda debe ejecutarse antes de abrir el notebook 04.**

In [ ]:
import joblib

paquete = {
    "X_train": X_train, "X_test": X_test,
    "y_train": y_train, "y_test": y_test,
    "cols_num": cols_num, "cols_cat": cols_cat,
    "top_paises": top_paises, "tope_adr": tope_adr,
}

destino = os.path.join(RUTA, "datos_preparados.joblib")
joblib.dump(paquete, destino)
print("Guardado en:", destino)
print("Tamano:", round(os.path.getsize(destino) / 1e6, 1), "MB")

---

**Siguiente paso:** `04_entrenamiento_y_evaluacion.ipynb`.